In [5]:
import cv2
import numpy as np
import math

cap = cv2.VideoCapture(0)

while True:
    success, frame = cap.read()

    if not success:
        break

    frame = cv2.flip(frame, 1)

    # bada hand detection area
    roi = frame[50:500, 50:500]

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    lower_skin = np.array([0, 30, 60])
    upper_skin = np.array([25, 255, 255])

    mask = cv2.inRange(hsv, lower_skin, upper_skin)

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.erode(mask, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=2)
    mask = cv2.GaussianBlur(mask, (7, 7), 0)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    finger_count = 0

    if len(contours) > 0:
        hand = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(hand)

        if area > 7000:
            hull = cv2.convexHull(hand)
            hull_index = cv2.convexHull(hand, returnPoints=False)

            cv2.drawContours(roi, [hand], -1, (0, 255, 0), 2)
            cv2.drawContours(roi, [hull], -1, (255, 0, 0), 2)

            try:
                defects = cv2.convexityDefects(hand, hull_index)

                if defects is not None:
                    gaps = 0

                    for i in range(defects.shape[0]):
                        s, e, f, d = defects[i][0]

                        start = tuple(hand[s][0])
                        end = tuple(hand[e][0])
                        far = tuple(hand[f][0])

                        a = math.dist(start, end)
                        b = math.dist(start, far)
                        c = math.dist(end, far)

                        angle = math.degrees(
                            math.acos((b*b + c*c - a*a) / (2*b*c))
                        )

                        if angle <= 90 and d > 10000:
                            gaps += 1
                            cv2.circle(roi, far, 5, (0, 0, 255), -1)

                    finger_count = gaps + 1

                    if finger_count > 5:
                        finger_count = 5

            except:
                finger_count = 0

    cv2.rectangle(frame, (0, 0), (250, 50), (0, 0, 0), -1)

    cv2.putText(
        frame,
        f"Fingers : {finger_count}",
        (10, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    # bada rectangle box
    cv2.rectangle(frame, (50, 50), (500, 500), (255, 0, 0), 3)

    cv2.imshow("Finger Counter", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

print("Final finger count:", finger_count)

cap.release()
cv2.destroyAllWindows()

Final finger count: 0
